# Bộ gác OCR · chọn ngưỡng trên keyframe thật

217 keyframe của 7 video L25 (đã OCR ở lần chạy 20261002). Khung có < 80 ký tự chữ là
**không có nội dung đọc được** (giảng viên nói, logo mở đầu). Bộ gác phải bỏ được nhiều khung
loại này nhất mà **không bỏ nhầm slide nào có chữ**.


In [ ]:
"""Shared artifact contract. Embedded into every notebook by build_notebooks.py."""

from pathlib import Path
import hashlib
import importlib.metadata
import json
import math
import re
import shutil
import subprocess
import threading
import unicodedata
from datetime import datetime, timezone

# transformers' from_pretrained builds models under a process-wide "meta device" context
# that is not thread-safe: two loads at once (OCR thread and main thread) fail with
# "Cannot copy out of meta tensor". Every model is loaded once per session, one at a time.
MODEL_LOAD_LOCK = threading.RLock()
_SHARED_MODELS = {}


def shared_model(key, loader):
    """Load a model once per session (reused by every video) and never two at once."""
    with MODEL_LOAD_LOCK:
        if key not in _SHARED_MODELS:
            _SHARED_MODELS[key] = loader()
        return _SHARED_MODELS[key]


SCHEMA_VERSION = "lecture-rag-v1"
PIPELINE_VERSION = "l25-v1"
STAGE_NAMES = {
    1: "01_slides", 2: "02_visual", 3: "03_audio",
    4: "04_bundle", 5: "05_index", 6: "06_retrieval",
}


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8",
    )
    temporary.replace(path)


def fingerprint(value):
    data = json.dumps(value, ensure_ascii=False, sort_keys=True, allow_nan=False)
    return hashlib.sha256(data.encode("utf-8")).hexdigest()


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def relative_asset(root, name):
    root = Path(root).resolve()
    path = (root / name).resolve()
    if not path.is_relative_to(root):
        raise ValueError(f"Asset nằm ngoài thư mục kết quả: {name}")
    return path


def run_command(arguments):
    process = subprocess.run(
        [str(value) for value in arguments], capture_output=True, text=True,
        encoding="utf-8", errors="replace",
    )
    if process.returncode:
        raise RuntimeError(
            f"{arguments[0]} thất bại ({process.returncode}):\n{process.stderr[-3000:]}"
        )
    return process.stdout


def package_versions(names):
    versions = {}
    for name in names:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = "not-installed"
    return versions


def find_video(override=None, input_root="/kaggle/input"):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(f"Không tìm thấy video: {path}")
        return path
    candidates = sorted(
        path for path in Path(input_root).rglob("*")
        if path.is_file()
        and path.suffix.lower() in {".mp4", ".mkv", ".mov", ".webm"}
        and path.stem.lower() in {"l25_v001", "tl25_v001"}
    )
    if len(candidates) != 1:
        raise ValueError(
            "Cần đúng một video L25_V001/TL25_V001. Add dataset "
            "thnhtrungnguynmif/l25a-lecture-videos-media hoặc đặt VIDEO_PATH. "
            f"Tìm thấy: {[str(path) for path in candidates]}"
        )
    return candidates[0]


def probe_video(path):
    if not shutil.which("ffprobe") or not shutil.which("ffmpeg"):
        raise RuntimeError("Cần FFmpeg và FFprobe trong PATH.")
    data = json.loads(run_command([
        "ffprobe", "-v", "error", "-show_format", "-show_streams",
        "-of", "json", str(path),
    ]))
    videos = [stream for stream in data["streams"] if stream["codec_type"] == "video"]
    audios = [stream for stream in data["streams"] if stream["codec_type"] == "audio"]
    if not videos:
        raise ValueError("Tệp không có luồng video.")
    info = data.get("format", {})
    origin = float(info.get("start_time", 0) or 0)
    duration = float(info.get("duration") or videos[0].get("duration") or 0)
    if not math.isfinite(duration) or duration <= 0:
        raise ValueError("Không đọc được thời lượng video hợp lệ.")
    return {
        "filename": Path(path).name,
        "sha256": file_sha256(path),
        "duration_sec": duration,
        "timeline_origin_sec": origin,
        "video_start_sec": float(videos[0].get("start_time", origin)),
        "audio_start_sec": float(audios[0].get("start_time", origin)) if audios else None,
        "has_audio": bool(audios),
        "width": int(videos[0]["width"]),
        "height": int(videos[0]["height"]),
        "clock": "original_presentation_timestamp_minus_format_start_time",
    }


def begin_stage(output_root, stage):
    directory = Path(output_root) / STAGE_NAMES[stage]
    directory.mkdir(parents=True, exist_ok=True)
    write_json(directory / "_RUNNING.json", {
        "stage": stage, "started_utc": datetime.now(timezone.utc).isoformat(),
    })
    return directory


def finish_stage(directory, stage, video_id, source, parameters, parents=(), extra=None):
    directory = Path(directory)
    files = {
        path.relative_to(directory).as_posix(): file_sha256(path)
        for path in sorted(directory.rglob("*"))
        if path.is_file()
        and path.name != "manifest.json"
        and not any(part.startswith("_") for part in path.relative_to(directory).parts)
        and path.suffix != ".tmp"
    }
    payload = {
        "schema_version": SCHEMA_VERSION,
        "pipeline_version": PIPELINE_VERSION,
        "stage": stage,
        "video_id": video_id,
        "source": source,
        "parameters": parameters,
        "parents": {str(parent["stage"]): parent["artifact_id"] for parent in parents},
        "files": files,
        **(extra or {}),
    }
    payload["artifact_id"] = fingerprint(payload)
    payload["created_utc"] = datetime.now(timezone.utc).isoformat()
    write_json(directory / "manifest.json", payload)
    (directory / "_RUNNING.json").unlink(missing_ok=True)
    print(f"Hoàn tất {STAGE_NAMES[stage]}: {directory}")
    return payload


def load_stage(directory, stage=None, video_id="L25_V001", verify=True):
    directory = Path(directory)
    if (directory / "_RUNNING.json").exists():
        raise ValueError(f"Bước xử lý chưa hoàn tất; cần chạy lại: {directory}")
    manifest = read_json(directory / "manifest.json")
    if manifest.get("schema_version") != SCHEMA_VERSION:
        raise ValueError(f"Schema không tương thích: {directory}")
    if stage is not None and manifest.get("stage") != stage:
        raise ValueError(f"Cần stage {stage}, nhận {manifest.get('stage')}: {directory}")
    if manifest.get("video_id") != video_id:
        raise ValueError(f"Sai video_id: {manifest.get('video_id')} != {video_id}")
    if manifest.get("pipeline_version") != PIPELINE_VERSION:
        raise ValueError(f"Sai pipeline_version: {directory}")
    signed = {key: value for key, value in manifest.items()
              if key not in {"artifact_id", "created_utc"}}
    if fingerprint(signed) != manifest["artifact_id"]:
        raise ValueError(f"Manifest đã thay đổi sau khi xuất: {directory}")
    if verify:
        for name, expected in manifest["files"].items():
            path = relative_asset(directory, name)
            if not path.is_file() or file_sha256(path) != expected:
                raise ValueError(f"Asset bị thiếu hoặc không khớp checksum: {path}")
    return manifest


def find_stage(stage, override=None, video_id="L25_V001",
               search_roots=("/kaggle/input", "/kaggle/working")):
    if override:
        directory = Path(override)
        return directory, load_stage(directory, stage, video_id)
    matches = []
    for root in map(Path, search_roots):
        if not root.exists():
            continue
        for path in root.rglob("manifest.json"):
            try:
                manifest = read_json(path)
            except (ValueError, OSError):
                continue
            if (manifest.get("schema_version") == SCHEMA_VERSION
                    and manifest.get("stage") == stage
                    and manifest.get("video_id") == video_id
                    and manifest.get("pipeline_version") == PIPELINE_VERSION):
                matches.append((path.parent, manifest))
    identities = {manifest.get("artifact_id") for _, manifest in matches}
    if not matches or len(identities) != 1:
        raise ValueError(
            f"Cần một phiên bản rõ ràng của {STAGE_NAMES[stage]}. "
            "Add Input → Notebook Output của bước trước, hoặc đặt STAGE_XX_DIR. "
            f"Các thư mục tìm được: {[str(path) for path, _ in matches]}"
        )
    directory = sorted(matches, key=lambda item: str(item[0]))[0][0]
    return directory, load_stage(directory, stage, video_id)


def require_same_source(*manifests):
    identities = {
        (item["video_id"], item["source"]["sha256"], item["pipeline_version"])
        for item in manifests
    }
    if len(identities) != 1:
        raise ValueError("Các nhánh không được tạo từ cùng video/pipeline.")


def require_parent(child, parent):
    require_same_source(child, parent)
    if child["parents"].get(str(parent["stage"])) != parent["artifact_id"]:
        raise ValueError(
            f"Stage {child['stage']} dùng phiên bản stage {parent['stage']} khác. "
            "Chạy lại nhánh với cùng kết quả phát hiện slide."
        )


def verify_intervals(intervals, duration, tolerance=0.03):
    if not intervals:
        raise ValueError("Danh sách interval rỗng.")
    ids = set()
    previous_end = 0.0
    for index, item in enumerate(intervals):
        start, end = float(item["start_sec"]), float(item["end_sec"])
        if not (math.isfinite(start) and math.isfinite(end)
                and 0 <= start < end <= duration + tolerance):
            raise ValueError(f"Interval không hợp lệ: {item}")
        if abs(start - previous_end) > tolerance:
            raise ValueError(f"Khoảng trống/chồng lấn ở interval {index}.")
        if item["segment_id"] in ids:
            raise ValueError("Trùng segment_id.")
        ids.add(item["segment_id"])
        previous_end = end
    if abs(previous_end - duration) > tolerance:
        raise ValueError("Interval cuối không bao phủ hết video.")


def owner_interval(intervals, time_sec):
    # Half-open intervals: a word exactly at a boundary belongs to the next one.
    for item in intervals:
        if item["start_sec"] <= time_sec < item["end_sec"]:
            return item
    if intervals and math.isclose(time_sec, intervals[-1]["end_sec"], abs_tol=1e-6):
        return intervals[-1]
    raise ValueError(f"Timestamp nằm ngoài lecture: {time_sec}")


def copy_stage_assets(source, destination, manifest):
    for name in manifest["files"]:
        target = relative_asset(destination, name)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(relative_asset(source, name), target)


def normalize_search(text):
    text = unicodedata.normalize("NFD", text.casefold().replace("đ", "d"))
    return "".join(char for char in text if not unicodedata.combining(char))


def timestamp_label(seconds):
    seconds = max(0, int(seconds))
    hours, seconds = divmod(seconds, 3600)
    minutes, seconds = divmod(seconds, 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}" if hours else f"{minutes:02}:{seconds:02}"


def unique_records(records, key):
    result = {}
    for record in records:
        if record[key] in result:
            raise ValueError(f"Trùng {key}: {record[key]}")
        result[record[key]] = record
    return result


def read_pcm_audio(path, sample_rate=16000):
    import numpy as np
    import wave

    with wave.open(str(path), "rb") as stream:
        if (stream.getframerate() != sample_rate or stream.getnchannels() != 1
                or stream.getsampwidth() != 2):
            raise ValueError("Audio phải là PCM signed 16-bit, mono, 16 kHz.")
        return np.frombuffer(stream.readframes(stream.getnframes()), dtype="<i2").copy()


def save_pcm_clip(path, pcm, start_sample, end_sample, sample_rate=16000):
    import wave

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with wave.open(str(path), "wb") as stream:
        stream.setnchannels(1)
        stream.setsampwidth(2)
        stream.setframerate(sample_rate)
        stream.writeframes(pcm[start_sample:end_sample].astype("<i2").tobytes())


In [ ]:
"""Ad filter: score each 1 Hz sample "lesson or ad", turn the scores into ad intervals and
cut them out of the slide intervals as their own segments (content_kind "ad").

Stage 01 calls apply_ad_filter() with the 1 Hz samples it already decoded. Later stages
skip "ad" segments: no OCR (02), their words stay with the ad (04, midpoint rule as for
slides), nothing is indexed (05).
"""

AD_LESSON_PROMPTS = [
    "a slide of a lecture presentation with text",
    "a lecture slide with a diagram, chart or formula",
    "a teacher presenting a lesson next to a slide",
    "the title slide of a lesson",
    "a page of a document or an exam with questions",
]
AD_PROMPTS = [
    "a video advertisement",
    "a promotional video of a school or university",
    "a logo animation",
    "students walking on a school campus",
    "people in a commercial",
    "a building exterior",
]


def ad_runs(mask):
    """[start, end) index runs of a 0/1 sequence where it is 1."""
    runs, index = [], 0
    while index < len(mask):
        if mask[index]:
            end = index
            while end < len(mask) and mask[end]:
                end += 1
            runs.append([index, end])
            index = end
        else:
            index += 1
    return runs


def ad_mask(scores, threshold=0.5, smooth=5, min_ad=5, max_gap=2):
    """Per-sample ad mask: moving average, threshold, fill short gaps between two ads,
    drop ads shorter than min_ad samples except a short closing logo at the very end."""
    import numpy as np

    scores = np.asarray(scores, dtype="float64")
    if smooth > 1 and len(scores):
        # edge padding: a short closing logo is not diluted by zeros past the end of the video
        padded = np.pad(scores, (smooth // 2, smooth - 1 - smooth // 2), mode="edge")
        scores = np.convolve(padded, np.ones(smooth) / smooth, mode="valid")
    mask = (scores >= threshold).astype(int)
    for start, end in ad_runs(1 - mask):
        if 0 < start and end < len(mask) and end - start <= max_gap:
            mask[start:end] = 1
    for start, end in ad_runs(mask):
        if end - start < min_ad and not (end == len(mask) and end - start >= 2):
            mask[start:end] = 0
    return mask


def model_features(output):
    """Embedding tensor from get_image_features/get_text_features (transformers 4 or 5)."""
    import torch

    return output if torch.is_tensor(output) else output.pooler_output


def load_siglip(model_id, device="cuda"):
    """SigLIP loaded once per session (shared by the ad filter and the OCR gate).
    Returns (embed_images(images), embed_texts(texts), logit_scale, logit_bias)."""
    def load():
        import torch
        from transformers import AutoModel, AutoProcessor

        dtype = torch.float16 if str(device).startswith("cuda") else torch.float32
        model = AutoModel.from_pretrained(model_id, torch_dtype=dtype).to(device).eval()
        processor = AutoProcessor.from_pretrained(model_id)

        def embed_images(images):
            with torch.no_grad():
                pixels = processor(images=images, return_tensors="pt")["pixel_values"].to(device, dtype)
                out = model_features(model.get_image_features(pixel_values=pixels)).float()
            return torch.nn.functional.normalize(out, dim=-1).cpu().numpy()

        def embed_texts(texts):
            with torch.no_grad():
                tokens = processor(text=texts, padding="max_length", return_tensors="pt").to(device)
                out = model_features(model.get_text_features(**tokens)).float()
            return torch.nn.functional.normalize(out, dim=-1).cpu().numpy()

        return embed_images, embed_texts, model.logit_scale.exp().item(), model.logit_bias.item()

    return shared_model(("siglip", model_id, str(device)), load)


def zero_shot(embed_images, embed_texts, scale, bias, positive, negative):
    """P(positive prompts) for each image: softmax over all prompts, summed over the positive ones."""
    import numpy as np

    text = embed_texts(list(positive) + list(negative))

    def score(images):
        logits = embed_images(images) @ text.T * scale + bias
        logits = logits - logits.max(1, keepdims=True)
        probs = np.exp(logits) / np.exp(logits).sum(1, keepdims=True)
        return probs[:, :len(positive)].sum(1).tolist()

    return score


def load_ad_scorer(mode="student", device="cuda", student_path=None,
                   teacher_model="google/siglip-so400m-patch14-384"):
    """Returns (score(images) -> list of P(ad), description). mode "teacher": SigLIP
    zero-shot with the prompts above; "student": logistic regression (student_path, an
    .npz from the ad filter experiments) on the image embeddings of a small SigLIP."""
    import numpy as np

    if mode not in {"student", "teacher"}:
        raise ValueError("AD_FILTER phải là 'student', 'teacher' hoặc None.")
    student = dict(np.load(student_path, allow_pickle=False)) if mode == "student" else None
    if student and bool(student.get("with_motion", False)):
        raise ValueError("Pipeline dùng model trò chỉ có ảnh (ad_student.npz), không phải bản có chuyển động.")
    model_id = str(student["backbone"]) if student else teacher_model
    embed_images, embed_texts, scale, bias = load_siglip(model_id, device)
    if mode == "teacher":
        score = zero_shot(embed_images, embed_texts, scale, bias, AD_PROMPTS, AD_LESSON_PROMPTS)
    else:
        def score(images):
            logits = embed_images(images) @ student["coef"] + float(student["intercept"][0])
            return (1 / (1 + np.exp(-logits))).tolist()
    threshold = float(student["threshold"]) if student else 0.5
    return score, {"mode": mode, "model": model_id, "threshold": threshold,
                   "student_sha256": file_sha256(student_path) if student else None}


# OCR gate: a keyframe goes to OCR only if it shows readable lesson content. A lecturer
# talking to the camera, an intro logo or a blank screen is kept as a segment (its speech
# stays in the lesson) but is not read by the slow OCR models.
CONTENT_PROMPTS = [
    "a slide of a lecture presentation with text",
    "a lecture slide with a diagram, chart, table or formula",
    "a page of a document or an exam with questions",
    "a whiteboard or blackboard with handwriting",
]
NO_CONTENT_PROMPTS = [
    "a person talking to the camera",
    "a teacher speaking in a studio",
    "a logo or a title animation",
    "a blank screen",
]


def load_ocr_gate(device="cuda", model_id="google/siglip-base-patch16-224", threshold=0.10):
    """Returns (check(images) -> list of P(readable content), description)."""
    embed_images, embed_texts, scale, bias = load_siglip(model_id, device)
    check = zero_shot(embed_images, embed_texts, scale, bias, CONTENT_PROMPTS, NO_CONTENT_PROMPTS)
    return check, {"model": model_id, "threshold": threshold}


def snap(time_sec, boundaries, tolerance=1.5):
    """Move an ad edge (from 1 Hz samples) to the nearest exact state boundary of stage 01."""
    nearest = min(boundaries, key=lambda item: abs(item - time_sec))
    return nearest if abs(nearest - time_sec) <= tolerance else time_sec


def ad_time_intervals(sample_times, mask, duration, boundaries):
    intervals = []
    for start, end in ad_runs(mask):
        a = snap(sample_times[start], boundaries)
        b = duration if end >= len(sample_times) else snap(sample_times[end], boundaries)
        if intervals and a <= intervals[-1][1]:
            intervals[-1][1] = max(intervals[-1][1], b)
        elif b > a:
            intervals.append([a, b])
    return intervals


def split_ad_intervals(intervals, ads, min_interval_sec=2.0):
    """Cut ad time ranges out of slide intervals. Returns [start, end, source] pieces in time
    order; source is the index of the slide interval, or None for an ad. A lesson remnant
    shorter than min_interval_sec next to an ad is absorbed by that ad."""
    cuts = sorted({item["start_sec"] for item in intervals} | {intervals[-1]["end_sec"]}
                  | {t for a, b in ads for t in (a, b) if intervals[0]["start_sec"] < t < intervals[-1]["end_sec"]})
    pieces = []
    for start, end in zip(cuts, cuts[1:]):
        middle = (start + end) / 2
        if any(a <= middle < b for a, b in ads):
            source = None
        else:
            source = next(i for i, item in enumerate(intervals) if item["start_sec"] <= middle < item["end_sec"])
        if pieces and pieces[-1][2] == source:
            pieces[-1][1] = end
        else:
            pieces.append([start, end, source])
    changed = True
    while changed:
        changed = False
        for i, (start, end, source) in enumerate(pieces):
            next_to_ad = (i > 0 and pieces[i - 1][2] is None) or (i + 1 < len(pieces) and pieces[i + 1][2] is None)
            if source is not None and end - start < min_interval_sec and next_to_ad:
                pieces[i][2] = None
                changed = True
        merged = []
        for piece in pieces:
            if merged and merged[-1][2] is None and piece[2] is None:
                merged[-1][1] = piece[1]
            else:
                merged.append(piece)
        pieces = merged
    return pieces


def apply_ad_filter(intervals, records, directory, video_id, duration, state_boundaries,
                    scorer, *, smooth=5, min_ad=5, max_gap=2, min_interval_sec=2.0,
                    batch_size=64):
    """intervals: stage-01 intervals (before writing). records: the 1 Hz samples of the scan
    (time_sec, image_path in directory). Returns (new intervals, ad report)."""
    from PIL import Image

    score, info = scorer
    times, scores = [item["time_sec"] for item in records], []
    for start in range(0, len(records), batch_size):
        images = []
        for item in records[start:start + batch_size]:
            with Image.open(Path(directory) / item["image_path"]) as opened:
                images.append(opened.convert("RGB"))
        scores += score(images)
    mask = ad_mask(scores, info["threshold"], smooth, min_ad, max_gap)
    ads = ad_time_intervals(times, mask, duration, sorted(set(state_boundaries)))
    pieces = split_ad_intervals(intervals, ads, min_interval_sec)

    result = []
    for start, end, source in pieces:
        number = len(result)
        if source is None:
            # representative frame: the 1 Hz sample nearest the middle of the ad
            middle = (start + end) / 2
            sample = min(records, key=lambda item: abs(item["time_sec"] - middle))
            name = f"keyframes/ad_{number:05d}.jpg"
            shutil.copyfile(Path(directory) / sample["image_path"], Path(directory) / name)
            result.append({
                "start_sec": round(start, 6), "end_sec": round(end, 6),
                "keyframe_time_sec": round(sample["time_sec"], 6), "image_path": name,
                "content_kind": "ad", "sharpness_score": round(sample["sharpness"], 3),
                "boundary_method": "ad_filter", "reveal_states": [],
            })
            continue
        item = dict(intervals[source])
        item["start_sec"], item["end_sec"] = round(start, 6), round(end, 6)
        if not start <= item["keyframe_time_sec"] < end:
            # the slide's own keyframe lies in the ad: use its latest state outside the ad
            inside = [state for state in item["reveal_states"]
                      if start <= state["keyframe_time_sec"] < end]
            if inside:
                state = inside[-1]
                item["keyframe_time_sec"], item["image_path"] = state["keyframe_time_sec"], state["image_path"]
            else:
                sample = min((r for r in records if start <= r["time_sec"] < end),
                             key=lambda r: -r["sharpness"], default=None)
                if sample is not None:
                    name = f"keyframes/trim_{number:05d}.jpg"
                    shutil.copyfile(Path(directory) / sample["image_path"], Path(directory) / name)
                    item["keyframe_time_sec"], item["image_path"] = sample["time_sec"], name
            item["reveal_states"] = [state for state in item["reveal_states"]
                                     if state["end_sec"] > start and state["start_sec"] < end]
        if (start, end) != (intervals[source]["start_sec"], intervals[source]["end_sec"]):
            item["boundary_method"] = item["boundary_method"] + "+ad_trim"
        result.append(item)
    for number, item in enumerate(result):
        item["segment_id"], item["sequence"] = f"{video_id}:seg_{number:05d}", number
        item["image_sha256"] = file_sha256(Path(directory) / item["image_path"])
    report = {**info, "smooth": smooth, "min_ad_sec": min_ad, "max_gap_sec": max_gap,
              "ad_intervals": [[round(a, 3), round(b, 3)] for a, b in ads],
              "ad_seconds": round(sum(b - a for a, b in ads), 1),
              "sample_scores": [round(value, 4) for value in scores]}
    return result, report


In [ ]:
import csv, glob
from pathlib import Path
from PIL import Image
base = Path(glob.glob('/kaggle/input/**/keyframes.csv', recursive=True)[0]).parent
rows = list(csv.DictReader(open(base / 'keyframes.csv', encoding='utf-8')))
images = [Image.open(base / r['image']).convert('RGB') for r in rows]
empty = [int(r['ocr_chars']) < 80 for r in rows]
print(len(rows), 'keyframe ·', sum(empty), 'không có nội dung')
results = {}
for model_id in ('google/siglip-base-patch16-224', 'google/siglip-so400m-patch14-384'):
    check, _ = load_ocr_gate('cuda', model_id)
    scores = []
    for i in range(0, len(images), 32):
        scores += check(images[i:i + 32])
    results[model_id] = scores
    print('\n' + model_id)
    for th in (0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7):
        skip = [s < th for s in scores]
        good = sum(s and e for s, e in zip(skip, empty))
        bad = sum(s and not e for s, e in zip(skip, empty))
        print(f'  ngưỡng {th:.2f}: bỏ đúng {good}/{sum(empty)} · bỏ nhầm slide có chữ {bad}/{len(rows) - sum(empty)}')
    content = sorted((s, r['image'], r['ocr_chars']) for s, r, e in zip(scores, rows, empty) if not e)
    print('  slide có chữ điểm thấp nhất:', [(round(s, 3), n, c) for s, n, c in content[:6]])
    print('  khung không nội dung điểm cao nhất:',
          sorted(((round(s, 3), r['image'], r['content_kind']) for s, r, e in zip(scores, rows, empty) if e), reverse=True)[:6])
import json
json.dump({'rows': rows, 'scores': results}, open('/kaggle/working/ocr_gate_scores.json', 'w'), ensure_ascii=False)
